# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `coldpink/Name  커넥트ai 장기메모리` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-4-E2B-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `coldpink/gemma-2b-brain-v4` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 246 · seq 1024 · linear · 양자화 q4_k_m (데이터 123개)


In [ ]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# 🔐 로그인을 맨 앞에서 확인 — 안 돼 있으면 긴 학습 전에 바로 멈춰서 시간 낭비 방지
from huggingface_hub import HfApi
try:
    print("✅ 로그인됨:", HfApi().whoami()["name"], "— 결과는 내 계정에 올라가요")
except Exception:
    raise SystemExit("❌ 먼저 위 🔑 칸에 HuggingFace write 토큰을 붙여넣고 Login을 누르세요. 그다음 [런타임 → 모두 실행]을 다시 누르면 됩니다.")


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


In [ ]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 (conversations Q&A)
내 지식이 **이 노트북에 직접 포함**돼 있어요 (업로드 불필요). 각 행 = `{conversations:[{user},{assistant}]}`


In [ ]:
import base64
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
# 내 지식(노트북에 포함) — base64로 안전하게 심어둠
_B64 = "..."
open("brain.jsonl", "w").write(base64.b64decode(_B64).decode("utf-8"))
ds = load_dataset("json", data_files="brain.jsonl", split="train")
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 246, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


In [ ]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 계열마다 다름 → 실제 텍스트에서 자동 감지 (gemma·llama·qwen 모두 지원)
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
if "<start_of_turn>user" in _t: _im, _rm = "<start_of_turn>user\n", "<start_of_turn>model\n"
elif "<|start_header_id|>" in _t: _im, _rm = "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
elif "<|im_start|>" in _t: _im, _rm = "<|im_start|>user\n", "<|im_start|>assistant\n"
elif "<|turn>user" in _t: _im, _rm = "<|turn>user\n", "<|turn>model\n"
else: _im, _rm = None, None
if _rm:
    trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
    print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 응답만 학습")
else:
    print("ℹ️ 마커 자동감지 실패 → 전체 텍스트로 학습(문제 없음)")


In [ ]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [ ]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    try:
        msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    except Exception:
        msg = [{"role":"user","content":prompt}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    inp = inp.to(model.device)
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


## 💾 저장 → HuggingFace
**safetensors(AI 진화·합성용) + GGUF(앱 실행용)** 둘 다 올라가요. (맨 앞에서 로그인했으니 바로 됩니다)


In [ ]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
# 📤 저장 위치 = "내" HF 계정 (위에서 로그인한 본인 계정으로 자동 — 노트북이 공유돼도 안전)
from huggingface_hub import HfApi
NAME = "gemma-2b-brain-v4"
OUTPUT = f'{HfApi().whoami()["name"]}/{NAME}'
print("📤 내 계정에 저장:", OUTPUT)
# ① 합성용 safetensors (AI 진화소에서 다시 합칠 수 있어요 — 이게 없으면 합성 불가!)
try:
    model.push_to_hub_merged(OUTPUT, tokenizer, save_method="merged_16bit", token=True)
    print("✅ safetensors 업로드 — AI 진화소에서 합치기 가능")
except Exception as e:
    print("⚠️ 병합 업로드 실패 → 어댑터(LoRA)로 폴백:", e)
    model.push_to_hub(OUTPUT, token=True); tokenizer.push_to_hub(OUTPUT, token=True)
# ② 앱 실행용 GGUF
model.push_to_hub_gguf(OUTPUT, tokenizer, quantization_method="q4_k_m", token=True)
print(f"✅ 완료! safetensors(합성용)+GGUF(실행용) 둘 다 → Connect AI 앱 🤖 내 AI 에서 {OUTPUT} 받기")
